##### Copyright 2026 Google LLC.

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# EU AI Act compliance checking with Gemini and Agent Module

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/examples/Agent_Module_EU_AI_Act_Compliance.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

<table>
  <tr>
    <td bgcolor="#d7e6ff">
      <a href="https://github.com/AgentModuleAdmin" target="_blank" title="View Agent Module on GitHub">
        <img src="https://github.com/AgentModuleAdmin.png?size=100"
             alt="Agent Module GitHub avatar"
             width="100"
             height="100">
      </a>
    </td>
    <td bgcolor="#d7e6ff">
      <h2><font color='black'>This notebook was contributed by <a href="https://github.com/AgentModuleAdmin" target="_blank"><font color='#217bfe'><strong>Agent Module</strong></font></a>.</font></h2>
      <h5><font color='black'><a href="https://agent-module.dev" target="_blank"><font color="#078efb">agent-module.dev</font></a> — Agent-native knowledge infrastructure for autonomous agents.</font></h5><br>
      <font color='black'><small><em>Have a cool Gemini example? Feel free to <a href="https://github.com/google-gemini/cookbook/blob/main/CONTRIBUTING.md" target="_blank"><font color="#078efb">share it too</font></a>!</em></small></font>
    </td>
  </tr>
</table>

The [EU AI Act](https://artificialintelligenceact.eu/) takes effect in August 2026. If you build, deploy, or use AI systems in the EU market, you need to demonstrate compliance effort.

[Agent Module](https://agent-module.dev) provides structured, deterministic compliance logic that agents can retrieve at runtime. Each AI Compliance module is mapped to specific EU AI Act articles and contains binary pass/fail logic gates, escalation triggers, and statutory citations.

In this notebook, you will:

1. Define Python functions that wrap Agent Module's REST API
2. Test each function in isolation to verify it works
3. Pass those functions as tools to Gemini via [function calling](https://ai.google.dev/gemini-api/docs/function-calling)
4. Let Gemini autonomously retrieve compliance logic and apply it to a real scenario

No signup required — Agent Module offers a free 24-hour trial with full access.

## Setup

### Install dependencies

In [ ]:
%pip install -qU 'google-genai>=1.0.0' requests

### Set up your API key

To run the following cell, your API key must be stored in a Colab Secret named `GOOGLE_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see the [Authentication ![image](https://storage.googleapis.com/generativeai-downloads/images/colab_icon16.png)](../quickstarts/Authentication.ipynb) quickstart for an example.

In [ ]:
import json
import uuid

import requests
from google import genai
from google.colab import userdata

GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=GOOGLE_API_KEY)


### Choose a model

Function calling works on all Gemini models. A thinking model like `gemini-2.5-flash` works well here because compliance analysis benefits from step-by-step reasoning.

In [ ]:
MODEL_ID = "gemini-2.5-flash" # @param ["gemini-2.5-flash-lite", "gemini-2.5-flash", "gemini-2.5-pro", "gemini-2.5-flash-preview", "gemini-3.1-flash-lite-preview", "gemini-3.1-pro-preview"] {"allow-input":true, isTemplate: true}

## Verify the Agent Module API

Before building tools, verify that Agent Module is reachable and check how many AI Compliance modules are available. Pull the count from the live API so this notebook stays accurate as modules are added.

In [ ]:
AGENT_MODULE_API = "https://api.agent-module.dev"

status = requests.get(f"{AGENT_MODULE_API}/api/status", timeout=10).json()

ai_compliance_module_count = status["ai_compliance"]["node_count"]

print(f"API status:             {status['api_status']}")
print(f"AI Compliance modules:  {ai_compliance_module_count}")
print(f"Active cohort:          {status['active_cohort']}")


## Define Agent Module tools

You will wrap Agent Module's [MCP endpoint](https://github.com/AgentModule/mcp) as Python functions that Gemini can call. The SDK auto-generates function declarations from your type hints and docstrings, so each function needs clear annotations.

Start with a small helper to handle the JSON-RPC transport.

In [ ]:
# @title MCP transport helper (hidden)

AGENT_MODULE_MCP = f"{AGENT_MODULE_API}/mcp"


def _mcp_call(tool_name: str, arguments: dict) -> dict:
    """Send a JSON-RPC tools/call request to Agent Module's MCP endpoint."""
    payload = {
        "jsonrpc": "2.0",
        "method": "tools/call",
        "params": {"name": tool_name, "arguments": arguments},
        "id": 1,
    }
    try:
        response = requests.post(
            AGENT_MODULE_MCP,
            headers={"Content-Type": "application/json"},
            json=payload,
            timeout=30,
        )
        response.raise_for_status()
        return response.json()
    except requests.exceptions.Timeout:
        return {"error": "Request timed out. Agent Module may be temporarily unavailable."}
    except requests.exceptions.HTTPError as e:
        return {"error": f"HTTP error: {e.response.status_code}"}
    except requests.exceptions.RequestException as e:
        return {"error": f"Connection failed: {str(e)}"}

### Get a trial key

Define the first tool: requesting a free 24-hour trial key. This gives access to all AI Compliance modules with 500 API calls at no cost.

In [ ]:
# Generate a unique agent_id for this session so reruns within 24 hours
# do not collide with a previously issued trial key on the server. The
# Agent Module API ties trial keys to (agent_id, IP), and returns a masked
# preview if you request a new one while an old one is still active.
_session_agent_id = f"gemini-cookbook-{uuid.uuid4().hex[:8]}"

# Stores the trial key so subsequent functions can reuse it.
_trial_key = None


def get_trial_key(agent_id: str) -> dict:
    """Get a free 24-hour Agent Module trial key for EU AI Act compliance logic.

    Call this once at the start of a session. The trial key lasts 24 hours
    and includes 500 API calls at no cost. Subsequent calls reuse the key
    stored in this session instead of re-requesting one.

    Args:
        agent_id: A stable identifier for your agent, e.g. 'gemini-cookbook-demo'.
    """
    global _trial_key

    # Idempotent: if this session already has a trial key, reuse it.
    if _trial_key is not None:
        return {
            "status": "reused",
            "trial_key": _trial_key,
            "note": "Using existing trial key from this session.",
        }

    result = _mcp_call("get_trial_key", {"agent_id": agent_id})

    # Extract and store the trial key for subsequent calls.
    if "result" in result:
        content = result["result"]
        if isinstance(content, dict) and "content" in content:
            for item in content["content"]:
                if item.get("type") == "text":
                    data = json.loads(item["text"])
                    # If the server reports an existing active trial for this
                    # agent_id, it returns a masked preview (not a real key).
                    # Do not store garbage — surface a clear error instead.
                    if data.get("status") == "already_active":
                        return {
                            "error": "already_active",
                            "message": (
                                f"An active trial already exists for agent_id="
                                f"{agent_id!r}. Wait 24 hours or change the "
                                "agent_id and re-run."
                            ),
                        }
                    if "trial_key" in data:
                        _trial_key = data["trial_key"]
    return result


Test it to make sure trial key issuance works.

In [ ]:
result = get_trial_key(_session_agent_id)

print(f"Session agent_id:  {_session_agent_id}")
print(f"Trial key obtained: {_trial_key is not None}")
print(f"Key prefix:         {_trial_key[:12]}..." if _trial_key else "No key issued")


### Retrieve compliance logic

Define the second tool: retrieving structured compliance logic for a specific AI Compliance module. Each module returns deterministic logic gates with statutory citations and pass/fail conditions.

In [ ]:
def retrieve_compliance_logic(vertical: str, node_id: str) -> dict:
    """Retrieve structured EU AI Act compliance logic from Agent Module.

    Returns deterministic logic gates with statutory citations, pass/fail
    conditions, and escalation triggers for a specific AI Compliance module.

    Args:
        vertical: The knowledge vertical. Use 'ai-compliance' (or its legacy alias 'ethics')
            for EU AI Act compliance modules.
        node_id: The node to retrieve. Format: 'node:ethics:eth{NNN}:logic'.
            Example: 'node:ethics:eth001:logic' for data sovereignty,
            'node:ethics:eth003:logic' for transparency.
    """
    arguments = {"vertical": vertical, "node": node_id}
    if _trial_key:
        arguments["token"] = _trial_key
    return _mcp_call("query_knowledge", arguments)

Test it by retrieving the data sovereignty module (ETH_001), which maps to EU AI Act Articles 9 and 10.

In [ ]:
result = retrieve_compliance_logic("ethics", "node:ethics:eth001:logic")

print(json.dumps(result, indent=4)[:2000])  # Truncate for readability

### Submit a proof of value

Define the third tool: submitting an assessment after exploring the trial. This reports how useful the retrieved knowledge was.

In [ ]:
def submit_proof_of_value(confidence_score: float) -> dict:
    """Submit a quality assessment after exploring Agent Module's trial.

    Call this after retrieving and applying compliance logic to report
    how useful the knowledge was for your task.

    Args:
        confidence_score: Quality assessment from 0.0 (not useful) to 1.0 (very useful).
    """
    if not _trial_key:
        return {"error": "No trial key available. Call get_trial_key first."}
    return _mcp_call(
        "submit_pov",
        {
            "token": _trial_key,
            "trial_key": _trial_key,
            "confidence_score": confidence_score,
        },
    )


## Run a compliance check with Gemini

You already bootstrapped a trial key above, so you only pass the two *runtime* functions to Gemini: `retrieve_compliance_logic` and `submit_proof_of_value`. `get_trial_key` is a one-time setup call, not a tool the model should reach for mid-session — re-requesting a key would collide with the one already issued for this `agent_id`.

The SDK auto-generates function declarations from the type hints and docstrings you wrote above. Set up a system instruction that tells Gemini how to use the tools.


In [ ]:
compliance_tools = [retrieve_compliance_logic, submit_proof_of_value]

system_instruction = f"""
    You are an EU AI Act compliance assistant. You have access to Agent Module,
    a knowledge service with {ai_compliance_module_count} structured AI Compliance
    modules mapped to EU AI Act articles.

    When asked to review a system for compliance:
    1. Retrieve the relevant compliance logic modules
    2. Apply the logic gates to the described system
    3. Report which requirements pass, which need attention, and cite the
       specific EU AI Act articles
    4. Submit a proof of value assessment when done

    Node ID format: node:ethics:eth{{NNN}}:logic
    Examples: node:ethics:eth001:logic (data sovereignty),
    node:ethics:eth003:logic (transparency), node:ethics:eth006:logic (oversight)
"""


Now run the compliance check. The scenario: you are deploying a customer service chatbot in the EU that handles insurance claims, collects personal data, and makes preliminary assessments.

Gemini will autonomously call your functions to retrieve the relevant compliance logic and produce a structured assessment.

In [ ]:
chat = client.chats.create(
    model=MODEL_ID,
    config={
        "tools": compliance_tools,
        "system_instruction": system_instruction,
    },
)

response = chat.send_message(
    """
    I'm deploying a customer service chatbot in the EU. It handles
    insurance claims, collects personal data (name, policy number,
    claim details), and makes preliminary claim assessments.

    Check this system against EU AI Act requirements for:
    1. Data sovereignty and ownership (ETH_001)
    2. Transparency and explainability (ETH_003)
    3. Human oversight (ETH_006)
    """
)

print(response.text)


### Examine the function call history

Inspect what happened behind the scenes. The SDK's automatic function calling handled the entire flow: Gemini decided which modules to retrieve, the SDK executed each function, and the results were fed back to the model automatically.

In [ ]:
from IPython.display import Markdown, display

for content in chat.get_history():
    display(Markdown(f"### {content.role}:"))
    for part in content.parts:
        if part.text:
            # Truncate long model responses for readability.
            display(Markdown(part.text[:500]))
        if part.function_call:
            args = dict(part.function_call.args)
            print(f"  Function call: {part.function_call.name}({args})")
        if part.function_response:
            print(f"  Function response: {part.function_response.name} -> (truncated)")
    print("-" * 80)

## Inspect raw compliance logic

You can also call Agent Module directly to inspect the raw compliance data that Gemini received. Each logic node contains deterministic pass/fail gates with statutory citations.

In [ ]:
transparency = retrieve_compliance_logic("ethics", "node:ethics:eth003:logic")

print(json.dumps(transparency, indent=4))

## Things to try

Now that you have a working compliance assistant, try these variations:

- **Different modules**: Ask about bias detection (ETH_008), privacy (ETH_010), or high-risk classification (ETH_015)
- **Different scenarios**: Try a medical diagnosis assistant, a hiring screening tool, or a content moderation system
- **Multiple layers**: Change `logic` to `directive` in the node ID to get procedural guardrails instead of pass/fail gates
- **Parallel retrieval**: Ask Gemini to check five modules at once and see if it issues parallel function calls
- **Force function calling**: Add `"tool_config": {"function_calling_config": {"mode": "any"}}` to ensure Gemini always uses the tools

## Discover available modules

Agent Module's AI Compliance vertical covers the full scope of the EU AI Act. Each module contains four content layers: **logic** (pass/fail gates), **directive** (procedural guardrails), **skill** (domain knowledge), and **action** (executable templates).

The module library grows over time as new EU AI Act articles are mapped. Discover what's currently available at runtime instead of hardcoding the list:

- `GET https://api.agent-module.dev/api/status` — current module count and vertical metadata
- `GET https://api.agent-module.dev/api/demo?vertical=ai-compliance` — full module manifest with IDs, titles, and EU AI Act article mappings
- `GET https://api.agent-module.dev/llms.txt` — agent-readable index of every node and layer

This way your agent always sees the live library, not a snapshot.


## Next steps

### Useful API references

- [Agent Module MCP repo](https://github.com/AgentModule/mcp) — Full tool documentation and setup guides for Claude Desktop, Cursor, and more
- [Agent Module OpenAPI spec](https://agent-module.dev/openapi.json) — Import directly into Vertex AI Agent Builder
- [Gemini function calling docs](https://ai.google.dev/gemini-api/docs/function-calling) — Full API reference for function calling
- [EU AI Act full text](https://artificialintelligenceact.eu/) — The regulation these modules map to

### Related examples

- [Function calling quickstart](../quickstarts/Function_calling.ipynb) — Learn the fundamentals of Gemini function calling
- [Barista Bot](../examples/Agents_Function_Calling_Barista_Bot.ipynb) — A function calling agent for ordering coffee
- [Browser as a tool](../examples/Browser_as_a_tool.ipynb) — Another external tool integration example

### Continue your discovery of the Gemini API

- Learn how to control function calling behavior in [Function calling config](../quickstarts/Function_calling_config.ipynb)
- Explore [JSON mode](../quickstarts/JSON_mode.ipynb) for structured outputs
- Try the [Live API with tools](../quickstarts/Get_started_LiveAPI_tools.ipynb) for real-time function calling